# Eksperimen 18 — Multi-view SBERT Retrieval + Reranking (Validation Only)

Eksperimen ini hanya menggunakan **validation v2 (38 query)**. Test 90 query yang sudah pernah dibuka **tidak digunakan** untuk memilih konfigurasi baru.

Tujuan eksperimen:

1. menambah jalur retrieval **ingredient-only SBERT** selain semantic retrieval lama (`title + ingredients`);
2. menguji reranking dengan tiga sinyal:
   - combined semantic similarity,
   - ingredient-only semantic similarity,
   - harmonic ingredient efficiency;
3. menguji tambahan **title ingredient anchor** yang hanya memakai query user dan judul kandidat pada runtime.

Tidak ada penggunaan `scraped_title` atau ground-truth title sebagai fitur runtime.

Baseline tetap:
- retrieval S50 + keyword max-20,
- min-max semantic,
- WLC 0.7 semantic + 0.3 harmonic efficiency.


In [1]:
import json
import os
import re

import numpy as np
import pandas as pd

from dapuria_pipeline import (
    DapuriaRecipeService,
    parse_user_ingredients,
    coverage_stats,
    harmonic_efficiency,
    ingredient_matches,
)

VAL_PATH = "ground_truth_relevance_validation_v2.csv"

ING_EMB_PATH = "ingredient_embeddings_5000.npy"
ING_IDS_PATH = "ingredient_embedding_ids.json"

SUMMARY_OUT = "hasil_multiview_sbert_validation_ringkas.csv"
DETAIL_OUT = "hasil_multiview_sbert_validation_detail.csv"

TOP_K_COMBINED = 50
TOP_K_INGREDIENT = 50
MAX_KEYWORD_EXTRA = 20


def split_pipe(x):
    if pd.isna(x):
        return []
    return [s.strip() for s in str(x).split(" | ") if s.strip()]


def hit_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    return int(any(rid in rel for rid in ranked_ids[:k]))


def precision_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    return sum(rid in rel for rid in ranked_ids[:k]) / k


def recall_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    if not rel:
        return 0.0
    return sum(rid in rel for rid in ranked_ids[:k]) / len(rel)


def reciprocal_rank(ranked_ids, relevant_ids):
    rel = set(relevant_ids)
    for rank, rid in enumerate(ranked_ids, start=1):
        if rid in rel:
            return 1.0 / rank
    return 0.0


def minmax(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) == 0:
        return arr
    lo = float(arr.min())
    hi = float(arr.max())
    return (arr - lo) / (hi - lo + 1e-9)


def title_anchor_score(user_ingredients, title):
    """
    Proporsi bahan user yang secara boundary-aware muncul pada judul kandidat.
    Hanya query user + candidate title. Tidak memakai ground-truth title.
    """
    if not user_ingredients:
        return 0.0

    matched = 0
    for u in user_ingredients:
        if ingredient_matches(u, title):
            matched += 1

    return matched / len(user_ingredients)


# ============================================================
# 1. Validation set
# ============================================================
val = pd.read_csv(VAL_PATH)

required = {"scraped_title", "query_text_v2", "relevant_ids"}
missing = required - set(val.columns)

if missing:
    raise ValueError(f"Kolom validation hilang: {sorted(missing)}")

dup = val.duplicated(["query_text_v2", "relevant_ids"]).sum()
if dup:
    raise ValueError(f"Validation masih punya {dup} duplicate cases.")

print(f"Validation cases             : {len(val)}")
print(f"Duplicate query+relevance   : {dup}")


# ============================================================
# 2. Load production service + corpus from Chroma
# ============================================================
service = DapuriaRecipeService(
    chroma_dir="chroma_db",
    collection_name="resep_masakan",
)

all_docs_raw = service.collection.get(include=["metadatas", "documents", "embeddings"])

all_ids = [str(x) for x in all_docs_raw["ids"]]
all_meta = all_docs_raw["metadatas"]
all_combined_emb = np.asarray(
    all_docs_raw["embeddings"],
    dtype=np.float32,
)

# Chroma embeddings were built normalized in the project.
# Normalize defensively again so dot-product remains cosine.
norms = np.linalg.norm(
    all_combined_emb,
    axis=1,
    keepdims=True,
)
all_combined_emb = all_combined_emb / np.clip(norms, 1e-12, None)

id_to_index = {rid: i for i, rid in enumerate(all_ids)}

parsed_meta = {}
ingredient_texts = []

for rid, meta in zip(all_ids, all_meta):
    p = service._parse_meta(meta)
    parsed_meta[rid] = p
    ingredient_texts.append(", ".join(str(x) for x in p["ingredient_names"]))

print(f"Corpus docs                  : {len(all_ids)}")


# ============================================================
# 3. Ingredient-only embeddings
# ============================================================
cache_ok = False

if os.path.exists(ING_EMB_PATH) and os.path.exists(ING_IDS_PATH):
    try:
        with open(ING_IDS_PATH, "r", encoding="utf-8") as f:
            cached_ids = json.load(f)

        cached_emb = np.load(ING_EMB_PATH)

        if cached_ids == all_ids and cached_emb.shape[0] == len(all_ids):
            ingredient_embeddings = cached_emb.astype(np.float32)
            cache_ok = True
            print("Ingredient embeddings        : loaded from cache")
    except Exception:
        cache_ok = False

if not cache_ok:
    print("Ingredient embeddings        : building 5,000 vectors...")

    ingredient_embeddings = service.model.encode(
        ingredient_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    np.save(
        ING_EMB_PATH,
        ingredient_embeddings,
    )

    with open(ING_IDS_PATH, "w", encoding="utf-8") as f:
        json.dump(all_ids, f, ensure_ascii=False)

    print("Ingredient embeddings        : saved to cache")


# ============================================================
# 4. Retrieval helpers
# ============================================================
def keyword_extra_candidates(
    user_ingredients,
    exclude_ids,
    max_extra=MAX_KEYWORD_EXTRA,
):
    scored = []

    for rid in all_ids:
        if rid in exclude_ids:
            continue

        names = parsed_meta[rid]["ingredient_names"]

        matched_count = sum(
            1 for u in user_ingredients if any(ingredient_matches(u, r) for r in names)
        )

        if matched_count <= 0:
            continue

        lexical_recipe_coverage = matched_count / len(names) if names else 0.0

        scored.append(
            (
                matched_count,
                lexical_recipe_coverage,
                rid,
            )
        )

    scored.sort(
        key=lambda x: (
            -x[0],
            -x[1],
            str(x[2]),
        )
    )

    return [x[2] for x in scored[:max_extra]]


def build_multiview_pool(query_text):
    user_ingredients = parse_user_ingredients(query_text)

    query_embedding = service.model.encode(
        [f"query: {query_text}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0].astype(np.float32)

    # Combined semantic similarity against stored title+ingredient vectors.
    combined_scores_all = all_combined_emb @ query_embedding

    combined_top_idx = np.argsort(-combined_scores_all)[:TOP_K_COMBINED]

    combined_top_ids = [all_ids[i] for i in combined_top_idx]

    # Ingredient-only semantic similarity.
    ingredient_query_embedding = service.model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0].astype(np.float32)

    ingredient_scores_all = ingredient_embeddings @ ingredient_query_embedding

    ingredient_top_idx = np.argsort(-ingredient_scores_all)[:TOP_K_INGREDIENT]

    ingredient_top_ids = [all_ids[i] for i in ingredient_top_idx]

    # Union semantic candidates.
    semantic_union = []
    seen = set()

    for rid in combined_top_ids + ingredient_top_ids:
        if rid not in seen:
            semantic_union.append(rid)
            seen.add(rid)

    # Keyword adds up to 20 unique candidates.
    keyword_ids = keyword_extra_candidates(
        user_ingredients,
        exclude_ids=seen,
        max_extra=MAX_KEYWORD_EXTRA,
    )

    pool_ids = semantic_union + keyword_ids

    prepared = []

    for rid in pool_ids:
        idx = id_to_index[rid]
        meta = parsed_meta[rid]

        stats = coverage_stats(
            user_ingredients,
            meta["ingredient_names"],
        )

        eff = harmonic_efficiency(
            stats["coverage_recipe"],
            stats["coverage_user"],
        )

        prepared.append(
            {
                "id": rid,
                "title": meta["title"],
                "ingredient_names": meta["ingredient_names"],
                "combined_raw": float(combined_scores_all[idx]),
                "ingredient_raw": float(ingredient_scores_all[idx]),
                "efficiency": float(eff),
                "title_anchor": float(
                    title_anchor_score(
                        user_ingredients,
                        meta["title"],
                    )
                ),
                "source_combined": int(rid in set(combined_top_ids)),
                "source_ingredient": int(rid in set(ingredient_top_ids)),
                "source_keyword": int(rid in set(keyword_ids)),
            }
        )

    # Normalize each semantic view inside the union pool.
    combined_mm = minmax([c["combined_raw"] for c in prepared])
    ingredient_mm = minmax([c["ingredient_raw"] for c in prepared])

    for c, s1, s2 in zip(
        prepared,
        combined_mm,
        ingredient_mm,
    ):
        c["combined_minmax"] = float(s1)
        c["ingredient_minmax"] = float(s2)

    return prepared, user_ingredients


# ============================================================
# 5. Baseline retrieval helper
# ============================================================
def build_baseline_pool(query_text):
    user_ingredients = parse_user_ingredients(query_text)

    candidates = service.retrieve_candidates(
        query_text,
        user_ingredients,
    )

    sem_mm = minmax([float(c["sbert_score"]) for c in candidates])

    prepared = []

    for c, s_norm in zip(candidates, sem_mm):
        stats = coverage_stats(
            user_ingredients,
            c["ingredient_names"],
        )

        eff = harmonic_efficiency(
            stats["coverage_recipe"],
            stats["coverage_user"],
        )

        prepared.append(
            {
                **c,
                "combined_minmax": float(s_norm),
                "ingredient_minmax": 0.0,
                "efficiency": float(eff),
                "title_anchor": float(
                    title_anchor_score(
                        user_ingredients,
                        c["title"],
                    )
                ),
            }
        )

    return prepared


# ============================================================
# 6. Cache validation pools
# ============================================================
cases = []

print("\nMembangun baseline + multiview pools...")

for _, row in val.iterrows():
    query = str(row["query_text_v2"]).strip()
    relevant_ids = set(split_pipe(row["relevant_ids"]))

    baseline_pool = build_baseline_pool(query)
    multiview_pool, user_ingredients = build_multiview_pool(query)

    cases.append(
        {
            "scraped_title": row["scraped_title"],
            "query": query,
            "relevant_ids": relevant_ids,
            "baseline_pool": baseline_pool,
            "multiview_pool": multiview_pool,
        }
    )

print("Pool cache selesai.")


# ============================================================
# 7. Configurations
# ============================================================
configs = [
    {
        "name": "BASELINE_07_03",
        "pool": "baseline_pool",
        "w_combined": 0.7,
        "w_ingredient": 0.0,
        "w_eff": 0.3,
        "w_anchor": 0.0,
    },
    # Multi-view without title anchor
    {
        "name": "MV_06_02_02",
        "pool": "multiview_pool",
        "w_combined": 0.6,
        "w_ingredient": 0.2,
        "w_eff": 0.2,
        "w_anchor": 0.0,
    },
    {
        "name": "MV_05_03_02",
        "pool": "multiview_pool",
        "w_combined": 0.5,
        "w_ingredient": 0.3,
        "w_eff": 0.2,
        "w_anchor": 0.0,
    },
    {
        "name": "MV_05_02_03",
        "pool": "multiview_pool",
        "w_combined": 0.5,
        "w_ingredient": 0.2,
        "w_eff": 0.3,
        "w_anchor": 0.0,
    },
    {
        "name": "MV_07_02_01",
        "pool": "multiview_pool",
        "w_combined": 0.7,
        "w_ingredient": 0.2,
        "w_eff": 0.1,
        "w_anchor": 0.0,
    },
    # Multi-view + runtime title anchor
    {
        "name": "MVA_05_02_02_01",
        "pool": "multiview_pool",
        "w_combined": 0.5,
        "w_ingredient": 0.2,
        "w_eff": 0.2,
        "w_anchor": 0.1,
    },
    {
        "name": "MVA_05_02_01_02",
        "pool": "multiview_pool",
        "w_combined": 0.5,
        "w_ingredient": 0.2,
        "w_eff": 0.1,
        "w_anchor": 0.2,
    },
    {
        "name": "MVA_06_015_015_01",
        "pool": "multiview_pool",
        "w_combined": 0.6,
        "w_ingredient": 0.15,
        "w_eff": 0.15,
        "w_anchor": 0.1,
    },
]


# ============================================================
# 8. Evaluate
# ============================================================
summary_rows = []
detail_rows = []

for config in configs:
    candidate_hits = []
    h5s = []
    h10s = []
    p5s = []
    p10s = []
    r5s = []
    r10s = []
    rrs = []
    pool_sizes = []

    for case in cases:
        pool = case[config["pool"]]
        relevant_ids = case["relevant_ids"]

        pool_ids = [str(c["id"]) for c in pool]
        pool_sizes.append(len(pool_ids))

        cand_hit = int(any(rid in relevant_ids for rid in pool_ids))

        ranked = []

        for c in pool:
            final_score = (
                config["w_combined"] * c["combined_minmax"]
                + config["w_ingredient"] * c["ingredient_minmax"]
                + config["w_eff"] * c["efficiency"]
                + config["w_anchor"] * c["title_anchor"]
            )

            ranked.append(
                {
                    **c,
                    "final_score_eval": float(final_score),
                }
            )

        ranked.sort(
            key=lambda x: (
                -x["final_score_eval"],
                -x["combined_minmax"],
                str(x["id"]),
            )
        )

        ranked_ids = [str(c["id"]) for c in ranked]

        h5 = hit_at_k(
            ranked_ids,
            relevant_ids,
            5,
        )
        h10 = hit_at_k(
            ranked_ids,
            relevant_ids,
            10,
        )
        p5 = precision_at_k(
            ranked_ids,
            relevant_ids,
            5,
        )
        p10 = precision_at_k(
            ranked_ids,
            relevant_ids,
            10,
        )
        r5 = recall_at_k(
            ranked_ids,
            relevant_ids,
            5,
        )
        r10 = recall_at_k(
            ranked_ids,
            relevant_ids,
            10,
        )
        rr = reciprocal_rank(
            ranked_ids,
            relevant_ids,
        )

        candidate_hits.append(cand_hit)
        h5s.append(h5)
        h10s.append(h10)
        p5s.append(p5)
        p10s.append(p10)
        r5s.append(r5)
        r10s.append(r10)
        rrs.append(rr)

        best_rank = None
        best_title = None

        for rank, c in enumerate(
            ranked,
            start=1,
        ):
            if str(c["id"]) in relevant_ids:
                best_rank = rank
                best_title = c["title"]
                break

        detail_rows.append(
            {
                "config": config["name"],
                "scraped_title": case["scraped_title"],
                "query_text": case["query"],
                "candidate_pool_size": len(pool),
                "candidate_hit": cand_hit,
                "best_relevant_rank": best_rank,
                "best_relevant_title": best_title,
                "hit_at_5": h5,
                "hit_at_10": h10,
                "precision_at_5": p5,
                "precision_at_10": p10,
                "recall_at_5": r5,
                "recall_at_10": r10,
                "reciprocal_rank": rr,
                "top10_titles": " | ".join(str(c["title"]) for c in ranked[:10]),
            }
        )

    summary_rows.append(
        {
            "config": config["name"],
            "pool_type": config["pool"],
            "w_combined": config["w_combined"],
            "w_ingredient": config["w_ingredient"],
            "w_eff": config["w_eff"],
            "w_anchor": config["w_anchor"],
            "n_query": len(cases),
            "avg_pool_size": float(np.mean(pool_sizes)),
            "candidate_hit_rate": float(np.mean(candidate_hits)),
            "candidate_hit_count": int(np.sum(candidate_hits)),
            "hit_at_5": float(np.mean(h5s)),
            "hit_at_5_count": int(np.sum(h5s)),
            "hit_at_10": float(np.mean(h10s)),
            "hit_at_10_count": int(np.sum(h10s)),
            "precision_at_5_macro": float(np.mean(p5s)),
            "precision_at_10_macro": float(np.mean(p10s)),
            "recall_at_5_macro": float(np.mean(r5s)),
            "recall_at_10_macro": float(np.mean(r10s)),
            "mrr": float(np.mean(rrs)),
        }
    )


summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(
        [
            "hit_at_5",
            "candidate_hit_rate",
            "hit_at_10",
            "mrr",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

detail_df = pd.DataFrame(detail_rows)

summary_df.to_csv(
    SUMMARY_OUT,
    index=False,
    encoding="utf-8-sig",
)

detail_df.to_csv(
    DETAIL_OUT,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 9. Print
# ============================================================
print("\n" + "=" * 120)
print("HASIL MULTI-VIEW SBERT - VALIDATION ONLY")
print("=" * 120)

print(summary_df.to_string(index=False))

best = summary_df.iloc[0]

baseline = summary_df[summary_df["config"] == "BASELINE_07_03"].iloc[0]

print("\nKONFIGURASI TERBAIK VALIDATION:")
print(
    f"{best['config']} | "
    f"CandidateHit={best['candidate_hit_rate']:.4f} "
    f"({int(best['candidate_hit_count'])}/{int(best['n_query'])}) | "
    f"Hit@5={best['hit_at_5']:.4f} "
    f"({int(best['hit_at_5_count'])}/{int(best['n_query'])}) | "
    f"Hit@10={best['hit_at_10']:.4f} | "
    f"MRR={best['mrr']:.4f}"
)

print("\nBASELINE:")
print(
    f"CandidateHit={baseline['candidate_hit_rate']:.4f} | "
    f"Hit@5={baseline['hit_at_5']:.4f} | "
    f"Hit@10={baseline['hit_at_10']:.4f} | "
    f"MRR={baseline['mrr']:.4f}"
)

print("\nPENTING:")
print("- Pemilihan konfigurasi hanya berdasarkan validation v2.")
print("- Test 90 query lama tidak dipakai untuk tuning.")
print(
    "- Ingredient-only embeddings disimpan sebagai cache agar run berikutnya lebih cepat."
)
print(
    "- Jika multiview tidak mengalahkan baseline, jangan tambahkan kompleksitas ini ke production."
)

print("\nOUTPUT:")
print(f"- {SUMMARY_OUT}")
print(f"- {DETAIL_OUT}")
print(f"- {ING_EMB_PATH}")
print(f"- {ING_IDS_PATH}")

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Validation cases             : 38
Duplicate query+relevance   : 0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6572.33it/s]


Corpus docs                  : 5000
Ingredient embeddings        : building 5,000 vectors...


Batches: 100%|██████████| 79/79 [00:53<00:00,  1.49it/s]


Ingredient embeddings        : saved to cache

Membangun baseline + multiview pools...
Pool cache selesai.

HASIL MULTI-VIEW SBERT - VALIDATION ONLY
           config      pool_type  w_combined  w_ingredient  w_eff  w_anchor  n_query  avg_pool_size  candidate_hit_rate  candidate_hit_count  hit_at_5  hit_at_5_count  hit_at_10  hit_at_10_count  precision_at_5_macro  precision_at_10_macro  recall_at_5_macro  recall_at_10_macro      mrr
  MVA_05_02_01_02 multiview_pool         0.5          0.20   0.10       0.2       38     109.578947            0.763158                   29  0.315789              12   0.394737               15              0.073684               0.044737           0.315789            0.368421 0.212040
MVA_06_015_015_01 multiview_pool         0.6          0.15   0.15       0.1       38     109.578947            0.763158                   29  0.315789              12   0.394737               15              0.073684               0.044737           0.302632            0.368